# Инициализация

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.patches import Patch
from IPython.display import display, clear_output
import ipywidgets as widgets
import random
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle

# Варьируемые параметры
max_val = 30          # максимальное значение при инициализации ячеек
min_val = 17           # минимальное значение при инициализации ячеек
GRID_SHAPE = (10, 14)  # Не квадрат! Можно менять на (7, 20), (14, 10), (15, 5)...
GRID_SIZE_Y, GRID_SIZE_X = GRID_SHAPE
enabled_random_options = True # Случайные опции

# Опции
OPTIONS = {
    'randomize_at_4': True,
    'neighbors_affect_speed': True,
    'even_steal_energy': True,
    'four_steals_neighbors': True,
    'odd_jump_if_surrounded_by_even': True,
    'use_8_neighbors': True,           # использовать 8 соседей (фон Нейман + диагонали)
    'only_even_update_if_r6': False,    # если есть ячейка из подмножества 6 ⇒ обновляются только чётные
    'only_odd_update_if_r3': False,     # если есть ячейка из подмножества 3 ⇒ обновляются только нечётные
    'even_numbers_add_2_if_sub6': False # если есть число из 6-го подпространства ⇒ чётные числа *2
}
# === Подписи на русском опций===
OPTION_LABELS = {
    'randomize_at_4': 'Обновление при достижении 4',
    'neighbors_affect_speed': 'Спуск ускоряется (рядом магистраль)',
    'even_steal_energy': 'Заморозка (рядом нечетные)',
    'four_steals_neighbors': 'Заморозка (рядом обновление)',
    'odd_jump_if_surrounded_by_even': '2 шага (рядом четные)',
    'use_8_neighbors': 'использовать 8 соседей (в ином случае 4)',
    'only_even_update_if_r6': 'Если есть ячейка из подмножества 6 ⇒ обновляются только чётные',
    'only_odd_update_if_r3': 'Если есть ячейка из подмножества 3 ⇒ обновляются только нечётные',
    'even_numbers_add_2_if_sub6': 'Если есть число из 6-го подпространства ⇒ чётные числа *2',
}

START_POS = (0, 0)
END_POS = (GRID_SIZE_Y - 1, GRID_SIZE_X - 1)
CYCLE_NUMBERS = {4, 2, 1}       # Числа из цикла
INITIAL_VALUES = [random.randint(min_val, max_val) for _ in range(GRID_SIZE_Y*GRID_SIZE_X)]
# === Специальные ячейки ===
START_FIN_CELL = -1       # Ячейки начала и конца для игры
STATIC_CELL_BOOST = -2    # Усиливающая ячейка
STATIC_CELL_EMPTY = -3    # Пустая ячейка
STATIC_CELL_CRYSTAL = -4  # Кристалл
STATIC_CELL_CRAFT = -5    # Крафтовая ячейка
# === Цвета специальных ячеек ===
COLOR_EMPTY = '#000000'       # Чёрный ⇒ пустое пространство
COLOR_BOOST_REGION = '#800080' # Пурпурный ⇒ область усиления
COLOR_CRYSTAL = '#00BFFF'  # Голубой ⇒ кристалл
COLOR_CRAFT_REGION = '#54B5F5' # крафтовая ячейка
COLOR_END_CELL = '#8E1F20' # Ячейки начала и конца для игры
COLOR_PREDATOR = '#FFFF00' # Противник (хищник)

# GAME
global player_circle, player_text, life_text, lives, game_over, use_shield, crystals_collected
global craft_cells_collected, active_craft_cell, craft_steps_counter, step_counter_predator

# ФЛАГИ
FLAG_FOG = False # Туман
FLAG_DEATH_MODE = True # Смертельный режим - 1 жизнь
FLAG_CELL_CRAFT = True # Включение крафтовых ячеек
FLAG_CELL_CRYSTALL = True # Включение ячеек кристалов
FLAG_CELL_EMPTY = True # Включение пустых ячеек
FLAG_CELL_BOOST = True # Включение ячеек с изменением закона

if FLAG_DEATH_MODE:
    lives = 3
player_circle = None
player_text = None
life_text = None
PLAYER_COLOR = 'blue'
PLAYER_RADIUS = 0.3
player_pos = list(START_POS)
crystals_collected = 0
craft_cells_collected = 0
active_craft_cell = None  # Текущая координата крафтовой ячейки, если игрок стоит на ней
craft_steps_counter = 0     # Счётчик шагов на крафтовой ячейке
step_counter_predator = 0
game_over = False
use_shield = False  # Защита действует один ход

def initialize_grid():
    """Инициализация решётки"""
    global crystal_coords
    grid = np.random.choice(INITIAL_VALUES, size=(GRID_SIZE_Y, GRID_SIZE_X))
    total_cells = GRID_SIZE_Y * GRID_SIZE_X

    # Заменяем стартовую и финишную ячейку на START_FIN_CELL
    grid[START_POS] = START_FIN_CELL
    grid[END_POS] = START_FIN_CELL

    # Изменениние закона
    boost_coords = []
    if FLAG_CELL_BOOST:
        num_boost_cells = int(total_cells * 0.03)  # 3%
        boost_coords = random.sample([(y, x) for y in range(GRID_SIZE_Y) for x in range(GRID_SIZE_X)
                                      if (y, x) not in [START_POS, END_POS]], num_boost_cells)
        for y, x in boost_coords:
            grid[y][x] = STATIC_CELL_BOOST

    # Пустые ячейки
    empty_coords = []
    if FLAG_CELL_EMPTY:
        num_empty_cells = int(total_cells * 0.04)  # 4%
        empty_coords = random.sample([(y, x) for y in range(GRID_SIZE_Y) for x in range(GRID_SIZE_X)
                                      if (y, x) not in boost_coords or
                                        (y, x) not in [START_POS, END_POS]], num_empty_cells)
        for y, x in empty_coords:
            grid[y][x] = STATIC_CELL_EMPTY

    # --- Кристалл: с вероятностью 75% создаём одну ячейку ---
    crystal_coords = []
    if FLAG_CELL_CRYSTALL:
        if random.random() < 0.75:
            possible_coords = [(y, x) for y in range(GRID_SIZE_Y) for x in range(GRID_SIZE_X)
                              if grid[y][x] > 0 or (y, x) not in boost_coords or
                                                    (y, x) not in empty_coords or
                                                    (y, x) not in [START_POS, END_POS]]
            if possible_coords:
                cy, cx = random.choice(possible_coords)
                grid[cy][cx] = STATIC_CELL_CRYSTAL
                crystal_coords.append((cy, cx))

    # Крафтовая ячейка
    craft_coords = []
    if FLAG_CELL_CRAFT:
        if random.random() < 0.75:
            possible_coords = [(y, x) for y in range(GRID_SIZE_Y) for x in range(GRID_SIZE_X)
                              if grid[y][x] > 0 or (y, x) not in crystal_coords or
                                                    (y, x) not in boost_coords or
                                                    (y, x) not in empty_coords or
                                                    (y, x) not in [START_POS, END_POS]]
            if possible_coords:
                cy, cx = random.choice(possible_coords)
                grid[cy][cx] = STATIC_CELL_CRAFT
                craft_coords.append((cy, cx))

    return grid


#------------------------function---------------------------------------
# === Проверка степени двойки ===
def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0

# === Закон Коллатца ===
def collatz_step(n, boost_neighbor):
    if n in CYCLE_NUMBERS:
        return n  # Цикл замыкается
    elif n % 2 == 0:
        return n // 2
    else:
        return pow(2, boost_neighbor)*(3 * n + 1)

# === Подпространства k для нечётных чисел ===
def get_subspace_k(n):
    if n % 2 == 0:
        return 0  # Чётное число ⇒ не имеет подпространства
    count = 0
    m = 3 * n + 1
    while m % 2 == 0:
        m //= 2
        count += 1
    return min(count, 6)

def update_options(options, prob=0.5):
    """Случайно включает/выключает опции"""
    return {k: v if random.random() < prob else not v for k, v in options.items()}

def check_cell_without_number(n):
    if n == START_FIN_CELL or n == STATIC_CELL_BOOST or n == STATIC_CELL_EMPTY or n == STATIC_CELL_CRAFT:
        return True
    else:
        return False

def return_neighbors(grid, y, x):
    neighbors = []
    if OPTIONS['use_8_neighbors']:
        dirs = [(-1, -1), (-1, 0), (-1, 1),
                (0, -1),          (0, 1),
                (1, -1), (1, 0), (1, 1)]
    else:
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for dy, dx in dirs:
        ny, nx = y + dy, x + dx
        if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X:
            neighbors.append(grid[ny][nx])

    has_boost_neighbor = False
    for dy, dx in dirs:
        ny, nx = y + dy, x + dx
        if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X:
            if grid[ny][nx] == STATIC_CELL_BOOST:
                has_boost_neighbor = True
                break

    return neighbors, has_boost_neighbor

def collatz_step_with_rules(n, boost_neighbor, min_val=5, max_val=85):
    # ---------------------------------------------------------
    if check_cell_without_number(n):
        return n, False  # Не обновляем и не замораживаем

    # --- Если число в цикле и включена опция randomize_at_4 ---
    if OPTIONS['randomize_at_4'] and n == 4:
        return np.random.randint(min_val, max_val + 1), False
    elif n == 4:
        return n, True

    # ---------------------------------------------------------

    # --- Если есть 6-е подпространство ⇒ чётные числа увеличиваются на 2 ---
    if OPTIONS['even_numbers_add_2_if_sub6'] and n % 2 == 0 and sub6_present:
        n = n * 2

    # ---------------------------------------------------------

    # --- Определяем, будет ли ячейка обновляться ---
    should_update = True
    if OPTIONS['only_even_update_if_r6'] and r6_present and OPTIONS['only_odd_update_if_r3'] and r3_present:
        should_update = True
    elif OPTIONS['only_even_update_if_r6'] and r6_present:
        should_update = (n % 2 == 0)  # Только чётные ячейки обновляются
    elif OPTIONS['only_odd_update_if_r3'] and r3_present:
        should_update = (n % 2 == 1)  # Только нечётные ячейки обновляются
    # --- Если обновление запрещено всеми условиями ⇒ пропускаем ---
    if not should_update:
        return n, True

    # --- Опция: если вокруг много нечётных, то чётная ячейка "не двигается" ---
    if OPTIONS['even_steal_energy']:
        odd_neighbors = sum(1 for m in neighbors if m % 2 == 1)
        if odd_neighbors > len(neighbors) / 2:
            return n, True

    # ---------------------------------------------------------

    # --- Опция: если соседи четные, то нечетная ячейка делает два шага ---
    if OPTIONS['odd_jump_if_surrounded_by_even']:
        even_neighbors = sum(1 for m in neighbors if m % 2 == 0)
        if even_neighbors > len(neighbors) / 2:
            n = collatz_step(collatz_step(n, boost_neighbor), boost_neighbor)

    # ---------------------------------------------------------

    # --- Опция: влияние магистрали ⇒ если рядом степень двойки ⇒ деление пополам два раза ---
    if OPTIONS['neighbors_affect_speed'] and n % 2 == 1:
        has_main_neighbor = any(is_power_of_two(m) and m >= 4 for m in neighbors)
        if has_main_neighbor:
            return collatz_step(collatz_step(n, boost_neighbor), boost_neighbor), False
        else:
            return collatz_step(n, boost_neighbor), False
    elif OPTIONS['neighbors_affect_speed'] and n % 2 == 0:
        has_main_neighbor = any(is_power_of_two(m) and m >= 4 for m in neighbors)
        if has_main_neighbor:
            return collatz_step(collatz_step(n, boost_neighbor), boost_neighbor), False
        else:
            return collatz_step(n, boost_neighbor), False
    else:
        return collatz_step(n, boost_neighbor), False

    return n, False

# === Обновление с опциями и заморозкой соседей ===
def update_grid_with_options(grid, frozen_cells=None):
    global r6_present, r3_present, sub6_present, neighbors
    new_grid = np.copy(grid)
    new_frozen = set()

    # --- Проверяем, есть ли ячейка из подмножества 6 ---
    r6_present = any(cell % 6 == 0 for row in grid for cell in row)
    # --- Проверяем, есть ли ячейка из подмножества 3 ---
    r3_present = any(cell % 6 == 3 for row in grid for cell in row)
    # --- Проверяем, есть ли 6-е подпространство ---
    sub6_present = any(get_subspace_k(cell) == 6 for row in grid for cell in row)

    for y in range(GRID_SIZE_Y):
        for x in range(GRID_SIZE_X):
            if frozen_cells and (y, x) in frozen_cells:
                continue  # Пропускаем замороженные ячейки

            n = grid[y][x]
            if check_cell_without_number(n):
                new_grid[y][x] = n  # Не меняем
            else:
                neighbors, has_boost_neighbor = return_neighbors(grid, y, x)

                boost_neighbor = 0
                if has_boost_neighbor:
                   boost_neighbor = 1;
                new_n, new_f = collatz_step_with_rules(n, boost_neighbor)
                if new_f:
                    new_frozen.add((y, x))
                # --- Опция: если число стало 4 и включено four_steals_neighbors ⇒ соседи замораживаются ---
                if OPTIONS['four_steals_neighbors'] and new_n == 4:
                    for dy, dx in [(-1,0), (1,0), (0,-1), (0,1)]:
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X:
                            new_frozen.add((ny, nx))

                new_grid[y][x] = new_n

    return new_grid, new_frozen


def predict_frozen_cells(grid):
    predicted_frozen = set()

    for y in range(GRID_SIZE_Y):
        for x in range(GRID_SIZE_X):
            n = grid[y][x]

            # Пропускаем статичные ячейки
            if check_cell_without_number(n):
                continue

            neighbors, has_boost_neighbor = return_neighbors(grid, y, x)

                # --- Если число в цикле и включена опция randomize_at_4 ---
            if OPTIONS['randomize_at_4'] and n == 4:
                True
            elif n == 4:
                predicted_frozen.add((y, x))

            # --- Определяем, будет ли ячейка обновляться ---
            should_update = True
            if OPTIONS['only_even_update_if_r6'] and r6_present and OPTIONS['only_odd_update_if_r3'] and r3_present:
                should_update = True
            elif OPTIONS['only_even_update_if_r6'] and r6_present:
                should_update = (n % 2 == 0)  # Только чётные ячейки обновляются
            elif OPTIONS['only_odd_update_if_r3'] and r3_present:
                should_update = (n % 2 == 1)  # Только нечётные ячейки обновляются
            # --- Если обновление запрещено всеми условиями ⇒ пропускаем ---
            if not should_update:
                predicted_frozen.add((y, x))

            # --- Опция: если вокруг много нечётных, то чётная ячейка "не двигается" ---
            if OPTIONS['even_steal_energy']:
                odd_neighbors = sum(1 for m in neighbors if m % 2 == 1)
                if odd_neighbors > len(neighbors) / 2:
                    predicted_frozen.add((y, x))

            # --- four_steals_neighbors: если ячейка станет 4 ⇒ заморозит соседей ---
            # Предполагаем, что она станет 4, если сейчас равна 2 или 8 (примеры)
            next_n = collatz_step(n, has_boost_neighbor)
            if OPTIONS['use_8_neighbors']:
                dirs = [(-1, -1), (-1, 0), (-1, 1),
                        (0, -1),          (0, 1),
                        (1, -1), (1, 0), (1, 1)]
            else:
                dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
            if OPTIONS['four_steals_neighbors'] and next_n == 4:
                for dy, dx in dirs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X:
                        predicted_frozen.add((ny, nx))

    return predicted_frozen

def steps_to_4(n):
    """Вычисляет, сколько шагов до числа 4"""
    if n in CYCLE_NUMBERS:
        return 0  # уже в цикле

    steps = 0

    while n not in CYCLE_NUMBERS:
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
        steps += 1

        if steps > 100000:  # защита от бесконечного цикла
            return -1

    return steps
#------------------------function---------------------------------------

# Игра

In [2]:
#-----------------------------game---------------------------------------------
option_widgets = {}
for key in OPTIONS:
    option_widgets[key] = widgets.Checkbox(
        value=OPTIONS[key],
        description=OPTION_LABELS[key],
        layout=widgets.Layout(width='auto')
    )
step_button = widgets.Button(description="Следующий шаг", layout=widgets.Layout(width='310px', height='30px'))
btn_frezze = widgets.Button(description="Заморозка ячейки", layout=widgets.Layout(width='310px', height='30px'))
btn_back = widgets.Button(description="Откат поля", layout=widgets.Layout(width='310px', height='30px'))
btn_up = widgets.Button(description="↑ Вверх", layout=widgets.Layout(width='310px', height='30px'))
btn_down = widgets.Button(description="↓ Вниз", layout=widgets.Layout(width='310px', height='30px'))
btn_left = widgets.Button(description="← Влево", layout=widgets.Layout(width='100px', height='30px'))
btn_right = widgets.Button(description="→ Вправо", layout=widgets.Layout(width='100px', height='30px'))
btn_stay = widgets.Button(description="🛑 Стоять", layout=widgets.Layout(width='100px', height='30px'))
btn_shield = widgets.Button(description="🛡️ Защита", layout=widgets.Layout(width='310px', height='30px'))
life_label = widgets.Label(value=f"❤️ Жизни: {lives}")
shield_label = widgets.Label(value="🛡️ Защита: ❌")
crystal_label = widgets.Label(value="💎 Собранные кристаллы: 0")
pres_label = widgets.Label(value="🗜 Допустимое давление: 200")
craft_label = widgets.Label(value="🛠️ Превращено крафтовых ячеек: 0")
frezze_label = widgets.Label(value="🥶 Заморозка ячейки: 0")
back_label = widgets.Label(value="🔙 Откат поля: 0")
info_label = widgets.Label(value="")
output_area = widgets.Output()

global predators
predators = []  # список координат хищников
def spawn_predator(grid):
    global predators
    if len(predators) >= 2:
        return grid

    empty_cells = [(y, x) for y in range(GRID_SIZE_Y) for x in range(GRID_SIZE_X)
                   if grid[y][x] > 0 and (y, x) not in predators and (y, x) != tuple(player_pos)]

    if random.random() < 1.0 and empty_cells:
        y, x = random.choice(empty_cells)
        grid[y][x], value_at_spawn = grid[y][x], grid[y][x]
        predators.append((y, x))
    return grid

# === Функция проверки состояния ===
def check_game_state():
    global lives, game_over, use_shield, crystals_collected, diamond, craft_cells_collected
    current_number = grids[0][player_pos[0]][player_pos[1]]
    current_position = tuple(player_pos)

    # Столкновение с хищником
    if current_position in predators:
        info_label.value = "💥 Вы встретили хищника ⇒ он уничтожён"
        predators.remove(current_position)
        # Заменяем ячейку на обычное число
        grids[0][current_position[0]][current_position[1]] = np.random.randint(17, 31)

    info_label.value = ""
    with output_area:
        clear_output(wait=True)
        if game_over:
            return

        if current_number == 4:
            info_label.value = "💥 Вы попали в число 4 ⇒ проигрыш!"
            game_over = True
            return
        elif current_number == STATIC_CELL_BOOST:
            info_label.value = "🌀 Вы вошли в аномальную зону усиления ⇒ вы погибли"
            game_over = True
            return
        elif current_number == STATIC_CELL_EMPTY:
            info_label.value = "⬛ Вы попали в пустоту ⇒ нельзя двигаться дальше"
            game_over = True
            return
        elif current_number == STATIC_CELL_CRYSTAL:
            crystals_collected += 1
            info_label.value = f"💎 Вы собрали кристалл! Всего: {crystals_collected}"
            crystal_label.value = f"💎 Собранные кристаллы: {crystals_collected}"
            grids[0][player_pos[0]][player_pos[1]] = random.randint(min_val, max_val)
            diamond.remove()
        elif current_number == STATIC_CELL_CRAFT:
            global active_craft_cell, craft_steps_counter

            if active_craft_cell is None or active_craft_cell != tuple(player_pos):
                # Только что вошёл на крафтовую ячейку
                active_craft_cell = tuple(player_pos)
                craft_steps_counter = 0
                info_label.value = "🛠️ Вы начали взаимодействовать с крафтовой ячейкой"
            else:
                craft_steps_counter += 1
                info_label.value = f"🛠️ Вы работаете над ячейкой... {craft_steps_counter}/3"

            # Подсчет окружения
            neighbors, _ = return_neighbors(grids[0], player_pos[0], player_pos[1])
            odd_count = sum(1 for n in neighbors if n % 2 == 1 if n >= 0)
            total = len(neighbors)

            damage = 0
            if odd_count >= total / 2:
                damage = 1
            if odd_count == total:
                damage = 2

            if damage > 0:
                if use_shield:
                    info_label.value += f"\n🛡️ Получили удар ({damage}), но защита спасла!"
                    use_shield = False
                    shield_label.value = "🛡️ Защита: ❌"
                else:
                    lives -= damage
                    info_label.value += f"\n💥 Получили {damage} урона из-за окружения"
                    life_label.value = f"❤️ Жизни: {lives}"
                    if lives <= 0:
                        info_label.value += "\n💀 Все жизни потеряны ⇒ проигрыш!"
                        game_over = True
                        return
            if craft_steps_counter >= 3:
                # Превращаем в случайное число
                new_value = random.randint(10, 80)
                grids[0][player_pos[0]][player_pos[1]] = new_value
                craft_cells_collected += 1
                active_craft_cell = None
                craft_steps_counter = 0
                craft_label.value = f"🛠️ Превращено крафтовых ячеек: {craft_cells_collected}"
                info_label.value = f"✅ Вы оживили крафтовую ячейку ⇒ стала числом {new_value}"

            return


        if current_number % 2 == 1 and not use_shield:
            lives -= 1

            info_label.value = f"⚠️ Нечётное число ⇒ потеряна жизнь! Осталось: {lives}"
            life_label.value = f"❤️ Жизни: {lives}"
            if lives <= 0:
                print("💀 Все жизни потеряны ⇒ проигрыш!")
                game_over = True
        elif current_number % 2 == 1 and use_shield:
            info_label.value = "✅ Защита активна ⇒ жизнь сохранена"
        else:
            info_label.value = "➡️ Число чётное ⇒ жизнь сохранена"

        use_shield = False
        shield_label.value = "🛡️ Защита: ❌"

        if player_pos == END_POS and grids[0][player_pos[0]][player_pos[1]] == START_FIN_CELL:
            # Проверяем, есть ли неактивированные крафтовые ячейки
            has_inactive_craft = any(cell == STATIC_CELL_CRAFT for row in grids[0] for cell in row)
            if has_inactive_craft:
                info_label.value = "⚠️ Нельзя завершить игру, пока остались неактивированные крафтовые ячейки"
            else:
                info_label.value = "🎉 Победа! Вы дошли до финиша"
                game_over = True


def step_and_move(dy, dx):
    global grid, player_pos, use_shield, player_circle, player_text
    if game_over:
        return

    # --- Сохраняем предыдущую позицию ---
    prev_y, prev_x = player_pos
    new_y = prev_y + dy
    new_x = prev_x + dx

    # --- Проверяем границы ---
    if 0 <= new_y < GRID_SIZE_Y and 0 <= new_x < GRID_SIZE_X:
        player_pos = [new_y, new_x]

    next_step()  # Обновляем матрицу

    # --- Удаляем старый кружок ---
    if player_circle:
        player_circle.remove()
    if player_text:
        player_text.remove()

    # --- Рисуем новый кружок на новой позиции ---
    current_number = grids[0][player_pos[0]][player_pos[1]]
    player_circle = plt.Circle((player_pos[1], player_pos[0]), PLAYER_RADIUS, color=PLAYER_COLOR, zorder=5)
    ax_main.add_patch(player_circle)

    # --- Текст внутри кружка ---
    fff =''
    if current_number > -1:
        fff = str(current_number)
    player_text = ax_main.text(player_pos[1], player_pos[0], fff,
                               ha='center', va='center', fontsize=8, color='white', zorder=6)

    check_game_state()
    draw_grid(predicted_frozen=predict_frozen_cells(grids[0]))

def toggle_shield(_):
    global use_shield
    if not game_over:
        use_shield = True
        shield_label.value = "🛡️ Защита: ✅"
    step_and_move(0, 0)

def move_up(_):
    step_and_move(1, 0)

def move_down(_):
    step_and_move(-1, 0)

def move_left(_):
    step_and_move(0, -1)

def move_right(_):
    step_and_move(0, 1)

def stay(_):
    step_and_move(0, 0)

btn_up.on_click(move_up)
btn_down.on_click(move_down)
btn_left.on_click(move_left)
btn_right.on_click(move_right)
btn_stay.on_click(stay)
btn_shield.on_click(toggle_shield)

def move_predators(grid):
    global predators, step_counter_predator
    if not predators or step_counter_predator % 2 != 0:
        return grid

    dirs = [(-1, -1), (-1, 0), (-1, 1),
            (0, -1),          (0, 1),
            (1, -1), (1, 0), (1, 1)]

    new_predators = []
    occupied = set(predators)

    for (py, px) in predators:
        best_move = None
        min_dist = float('inf')

        for dy, dx in dirs:
            ny, nx = py + dy, px + dx
            if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X and (ny, nx) not in occupied:
                dist = abs(ny - player_pos[0]) + abs(nx - player_pos[1])
                if dist >= 2:  # Не приближаться ближе чем на 1 клетку
                    if dist < min_dist:
                        min_dist = dist
                        best_move = (ny, nx)

        if best_move:
            # Обмениваем местами значения ячеек
            grid[py][px], grid[best_move[0]][best_move[1]] = grid[best_move[0]][best_move[1]], grid[py][px]
            new_predators.append(best_move)
        else:
            new_predators.append((py, px))  # Невозможно двигаться — остаётся на месте

    predators = new_predators

def destroy_predators_near_4(grid):
    global predators
    to_remove = set()
    dirs = [(-1, -1), (-1, 0), (-1, 1),
            (0, -1),          (0, 1),
            (1, -1), (1, 0), (1, 1)]

    for i, (py, px) in enumerate(predators):
        for dy, dx in dirs:
            ny, nx = py + dy, px + dx
            if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X:
                if grid[ny][nx] == 4:
                    to_remove.add(i)

    # Удаляем хищников
    for i in sorted(to_remove, reverse=True):
        py, px = predators[i]
        grid[py][px] = np.random.randint(17, 31)
        del predators[i]

def next_step():
    global grids, frozen_cells_global, step_counter_predator
    if game_over:
        return

    grid = grids[0]  # Берём основную решётку для обработки

    # --- Обновляем решётку и получаем реальные заморозки ---
    new_grid, new_frozen = update_grid_with_options(grid, frozen_cells_global)
    destroy_predators_near_4(new_grid)
    grids[0] = new_grid
    frozen_cells_global = new_frozen

    # --- Прогнозируем заморозки до обновления ---
    predicted_frozen = predict_frozen_cells(grid)

    # --- Рисуем с прогнозом заморозок на следующий ход ---
    draw_grid(predicted_frozen=predicted_frozen)

    step_counter_predator += 1
    move_predators(grids[0])
    grids[0] = spawn_predator(grids[0])

# === Кнопка следующего шага ===
def on_button_clicked(b):
    next_step()

step_button.on_click(on_button_clicked)

# === Обновление опций ===
def make_checkbox_handler(key):
    def handler(change):
        OPTIONS[key] = change['new']
    return handler
#-----------------------------game---------------------------------------------

# Графики

In [3]:
def draw_predator(ax, grid):
    for (y, x) in predators:
        predator_circle = plt.Circle((x, y), PLAYER_RADIUS, color=COLOR_PREDATOR, zorder=5)
        ax.add_patch(predator_circle)
        # Показываем число, которое было в ячейке до появления хищника
        cell_value = grid[y][x]
        text = ax.text(x, y, str(cell_value), ha='center', va='center',
                       fontsize=8, color='black', zorder=6)

#------------------------COLOR---------------------------------------
def check_active_rules(grid, n, y, x):
    """Проверяет, какие правила сейчас влияют на ячейку"""
    rules = []
    four_present = (n == 4)

    neighbors, _ = return_neighbors(grid, y, x)

    if OPTIONS['randomize_at_4'] and n == 4:
        rules.append('randomize_at_4')

    if OPTIONS['neighbors_affect_speed']:
        has_main_neighbor = any(is_power_of_two(m) and m >= 4 for m in neighbors)
        if has_main_neighbor:
            rules.append('neighbors_affect_speed')

    if OPTIONS['even_steal_energy']:
        odd_neighbors = sum(1 for m in neighbors if m % 2 == 1)
        if odd_neighbors > len(neighbors) / 2:
            rules.append('even_steal_energy')

    if OPTIONS['four_steals_neighbors'] and n == 4:
        rules.append('four_steals_neighbors')

    if OPTIONS['odd_jump_if_surrounded_by_even']:
        even_neighbors = sum(1 for m in neighbors if m % 2 == 0)
        if even_neighbors > len(neighbors) / 2:
            rules.append('odd_jump_if_surrounded_by_even')

    if OPTIONS['only_even_update_if_r6'] and r6_present:
        rules.append('r6_lock')
    if OPTIONS['only_odd_update_if_r3'] and r3_present:
        rules.append('r3_lock')
    if OPTIONS['even_numbers_add_2_if_sub6'] and sub6_present and n % 2 == 0:
        rules.append('sub6_boost')
    if OPTIONS['four_steals_neighbors'] and four_present:
        rules.append('four_freeze')
    # print(f"Ячейка ({y},{x}) число {n} имеет правила: {rules}")  # Отладка
    return rules

def get_color_cell_without_number_2(n):
    color = {
        START_FIN_CELL: COLOR_END_CELL,
        STATIC_CELL_BOOST: COLOR_BOOST_REGION,
        STATIC_CELL_EMPTY: COLOR_EMPTY,
        STATIC_CELL_CRYSTAL: COLOR_CRYSTAL,
        STATIC_CELL_CRAFT: COLOR_CRAFT_REGION
    }.get(int(n), 'white')
    return color

def get_color_cell_without_number(n, y, x):
    color = get_color_cell_without_number_2(n)
    rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                          facecolor=color, edgecolor='black', linewidth=1)
    return rect

#     return base_rect  # Возвращаем основной прямоугольник как "контейнер"
def get_rule_color(n, grid, y, x):
    """Возвращает прямоугольник с полосами по активным правилам"""
    if check_cell_without_number(n):
        return get_color_cell_without_number(n, y, x)

    active_rules = check_active_rules(grid, n, y, x)

    # Если нет правил ⇒ цвет по типу числа
    if not active_rules:
        if n in CYCLE_NUMBERS:
            rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                                 facecolor="#FFD700", edgecolor='black', linewidth=1)
        elif n % 2 == 0:
            rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                                 facecolor="#F0F0F0", edgecolor='black', linewidth=1)
        else:
            rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                                 facecolor="#E0E0E0", edgecolor='black', linewidth=1)
        return rect

    # === Есть правила ⇒ рисуем полосы ===
    num_rules = len(active_rules)
    bar_width = 1 / num_rules

    rects = []

    # Базовый фон (прозрачный или белый) — не обязателен, если есть полосы
    # Вместо базового фона — каждая полоса заполняет пространство
    for i, rule in enumerate(active_rules):
        color = {
            'randomize_at_4': "#FF0000",
            'neighbors_affect_speed': "#00FF00",
            'even_steal_energy': "#0000FF",
            'four_steals_neighbors': "#FFFF00",
            'odd_jump_if_surrounded_by_even': "#FFA500",
            'r6_lock': '#FF6347',
            'r3_lock': '#4682B4',
            'sub6_boost': '#32CD32',
            'four_freeze': '#BA55D3'
        }.get(rule, "#888888")

        # Каждая полоса занимает почти весь размер ⇒ добавляем небольшие промежутки для белой линии
        spacing = 0.01  # ширина белой линии между полосами
        actual_width = bar_width - spacing

        bar_rect = plt.Rectangle(
            (x - 0.5 + i * bar_width + spacing / 2, y - 0.5),
            actual_width,
            1,
            facecolor=color,
            edgecolor='white',     # Линия между полосами — белая
            linewidth=1            # Толщина белых линий
        )
        rects.append(bar_rect)

    # Чтобы избежать наложения, можно добавить обводку вокруг всей ячейки
    border = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                           fill=False, edgecolor='black', linewidth=1)

    rects.append(border)  # Чёрная граница вокруг ячейки

    # Для корректной отрисовки возвращаем список патчей
    return rects

# === Цвета ===
ARM_COLORS = {
    0: '#FFA07A',  # LightSalmon
    1: '#98FB98',  # PaleGreen
    2: '#87CEEB',  # SkyBlue
    3: '#FFB6C1',  # LightPink
    4: '#FFFFFF',  # Белый
    5: '#22D755'   #
}
# === Создание цветовой карты для 1 картинки ===
def get_arm_color(n, grid=None, y=None, x=None):
    if check_cell_without_number(n):
        return get_color_cell_without_number(n, y, x)

    color = None
    r = n % 6
    color = ARM_COLORS.get(r, '#CCCCCC')  # По умолчанию нейтральный цвет
    rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
    return rect

def get_freeze_color(val, grid=None, y=None, x=None):
    color = None
    if val == True:
        color = "#0000AA"
    else:
        color = "#00AA00"
    rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
    return rect

EVEN_ODD_COLORS = {
    "odd": "#4682B4",   # Стальные для нечётных
    "even": "#FFFFFF",  # Хаки для чётных
    "cycle": "#FFD700"  # Белый для цикла 4 → 2 → 1
}
# === Получение цвета по чётности ===
def get_even_odd_color(n, grid=None, y=None, x=None):
    if check_cell_without_number(n):
        return get_color_cell_without_number(n, y, x)

    color = None
    if n % 2 == 1:
        color = EVEN_ODD_COLORS["odd"]
    else:
        color = EVEN_ODD_COLORS["even"]
    rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
    return rect

SUBSPACE_COLORS = {
    1: "#FF6347",   # Томатный
    2: "#4682B4",  # Стальной
    3: "#32CD32",  # Лаймовый
    4: "#4169E1",  # Индиго
    5: "#BA55D3",  # Орхидея
    6: "#338C00",  #
    "even": "#FFFFFF",
    "cycle": "#FFD700"  # Жёлтый ⇒ числа из цикла (4, 2, 1)
}
# === Получение цвета по подпространству ===
def get_subspace_color(n, grid=None, y=None, x=None):
    if check_cell_without_number(n):
        return get_color_cell_without_number(n, y, x)

    color = None
    if n % 2 == 1:
        k = get_subspace_k(n)
        color = SUBSPACE_COLORS.get(k, "#CCCCCC")
    else:
        color = SUBSPACE_COLORS["even"]
    rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
    return rect

# --- главный график: подпространства + четность + цикл + подмножества ---
def get_combined_color(n, grid=None, y=None, x=None):
    if check_cell_without_number(n):
        return get_color_cell_without_number(n, y, x)

    color = None
    if n in CYCLE_NUMBERS:
        color = "#FFD700"  # Жёлтый ⇒ часть цикла
    elif n % 2 == 0:
        color = "#F0F0F0"  # Светло-серый ⇒ чётное число вне цикла
    else:
        k = get_subspace_k(n)
        if k == 1:
            color = "#FF6347"  # Томатный ⇒ подпространство 1 (возможный рост)
        elif 2 <= k <= 6:
            color = "#32CD32"  # Лайм ⇒ подпространства 2–6 (спуск)
        else:
            color = "#CCCCCC"  # Нейтральный ⇒ не определено
    rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
    return rect

def get_steps_color(n, steps, y=None, x=None):
    """Возвращает цвет, соответствующий количеству шагов до числа 4"""
    if check_cell_without_number(n):
        return get_color_cell_without_number_2(n)

    if steps == -1:
        return "#AAAAAA"  # Серый ⇒ если не сошлось за 1000 шагов
    elif steps == 0:
        return "#FFD700"  # Жёлтый ⇒ уже в цикле
    else:
        max_steps = 20
        intensity = min(steps, max_steps)
        gray_value = int(255 * (1 - intensity / max_steps))
        hex_gray = f"{gray_value:02X}"
        return f"#{hex_gray}{hex_gray}FF"  # Градиент от синего до серого
#------------------------COLOR---------------------------------------



#------------------------graph---------------------------------------
def initialize_graphics_arrays():
    """Инициализирует переменные для графиков"""
    global arm_cells, eo_cells, sub_cells, main_cells, combined_cells, freeze_cells, step_cells
    global arm_texts, eo_texts, sub_texts, main_texts, combined_texts, freeze_texts, step_texts

    # Инициализация под текущую форму поля
    height, width = GRID_SIZE_Y, GRID_SIZE_X

    # Ячейки
    arm_cells = [[None for _ in range(width)] for _ in range(height)]
    eo_cells = [[None for _ in range(width)] for _ in range(height)]
    sub_cells = [[None for _ in range(width)] for _ in range(height)]
    main_cells = [[None for _ in range(width)] for _ in range(height)]
    combined_cells = [[None for _ in range(width)] for _ in range(height)]
    freeze_cells = [[None for _ in range(width)] for _ in range(height)]
    step_cells = [[None for _ in range(width)] for _ in range(height)]

    # Тексты
    arm_texts = [[None for _ in range(width)] for _ in range(height)]
    eo_texts = [[None for _ in range(width)] for _ in range(height)]
    sub_texts = [[None for _ in range(width)] for _ in range(height)]
    main_texts = [[None for _ in range(width)] for _ in range(height)]
    combined_texts = [[None for _ in range(width)] for _ in range(height)]
    freeze_texts = [[None for _ in range(width)] for _ in range(height)]
    step_texts = [[None for _ in range(width)] for _ in range(height)]

def draw_crystal(ax, x, y):
    global diamond
    diamond = plt.Polygon([
        (x - 0.25, y),
        (x, y - 0.25),
        (x + 0.25, y),
        (x, y + 0.25)
    ], closed=True, facecolor='cyan', edgecolor='black', linewidth=1, zorder=4)
    ax.add_patch(diamond)

def draw_handled(ax, grid_data, color_func, text_array, cell_array):
    ax.clear()
    ax.set_xticks([])
    ax.set_yticks([])
    height, width = grid_data.shape
    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(-0.5, height - 0.5)
    ax.set_aspect('equal')

    for y in range(height):
        for x in range(width):
            value = grid_data[y][x]

            # Получаем объект(ы) из color_func
            result = color_func(value, grid_data, y, x)

            # Если результат — список ⇒ это многослойный Rect (полосы)
            if isinstance(result, list):
                for r in result:
                    ax.add_patch(r)
                cell_array[y][x] = result[0]
            else:
                # Одиночный Rectangle
                ax.add_patch(result)
                cell_array[y][x] = result

            # Убираем текст на спецячейках
            txt = None
            if check_cell_without_number(value):
                True
            else:
                txt = ax.text(x, y, str(value), ha='center', va='center', fontsize=8, color="black")
            text_array[y][x] = txt

def draw_steps(ax, grid_data, text_array, cell_array):
    ax.clear()
    ax.set_xticks([])
    ax.set_yticks([])
    height, width = grid_data.shape
    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(-0.5, height - 0.5)
    ax.set_aspect('equal')

    for y in range(height):
        for x in range(width):
            value = grid_data[y][x]
            steps = steps_to_4(value)
            color = get_steps_color(value, steps, y, x)
            rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                                 facecolor=color, edgecolor='black', linewidth=1)
            cell_array[y][x] = rect
            ax.add_patch(rect)

            # Отображаем число шагов, а не само число
            txt = None
            if check_cell_without_number(value):
                True
            else:
                txt = ax.text(x, y, str(steps), ha='center', va='center', fontsize=8, color="black")
            text_array[y][x] = txt

def draw_freeze_map(ax, grid, frozen_set):
    ax.clear()
    ax.set_xticks([])
    ax.set_yticks([])

    height, width = grid.shape
    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(-0.5, height - 0.5)
    ax.set_aspect('equal')

    for y in range(height):
        for x in range(width):
            value = grid[y][x]
            is_frozen = (y, x) in frozen_set

            if value == START_FIN_CELL:
                color = 'black'
            elif value == STATIC_CELL_EMPTY:
                color = COLOR_EMPTY
            elif value == STATIC_CELL_BOOST:
                color = COLOR_BOOST_REGION
            else:
                color = "#0000AA" if is_frozen else "#00AA00"

            rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                                 facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)

            # Показываем число или ❄️, если заморожено
            if is_frozen:
                ax.text(x, y, '❄️', ha='center', va='center', fontsize=8, color='white')
            elif check_cell_without_number(value):
                True
            else:
                ax.text(x, y, str(value), ha='center', va='center', fontsize=8, color='black')

# --- Рисуем туман войны ---
def draw_fog_of_war(ax, grid, player_y, player_x):
    # Определяем соседей игрока
    if OPTIONS['use_8_neighbors']:
        dirs = [(-1, -1), (-1, 0), (-1, 1),
                (0, -1),          (0, 1),
                (1, -1), (1, 0), (1, 1)]
    else:
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    visible_cells = [(player_y, player_x)]
    for dy, dx in dirs:
        ny, nx = player_y + dy, player_x + dx
        if 0 <= ny < GRID_SIZE_Y and 0 <= nx < GRID_SIZE_X:
            visible_cells.append((ny, nx))

    # Прорезаем "дыры" для видимых ячеек
    for y in range(GRID_SIZE_Y):
        for x in range(GRID_SIZE_X):
            if (y, x) in visible_cells:
                True
            else:
                fog = plt.Rectangle((x - 0.5, y - 0.5), 1, 1,
                                    facecolor='black', alpha=1.0, zorder=3)
                ax.add_patch(fog)

def draw_grid(predicted_frozen=None):
    global grids, ax_arm, ax_eo, ax_sub, ax_rule, ax_main, ax_freeze, ax_steps
    global legend_arm, legend_eo, legend_sub, legend_rule, legend_main, legend_freeze, legend_steps
    global life_text, player_circle, player_text, crystal_coords

    global r6_present, r3_present, sub6_present
    # --- Проверяем, есть ли ячейка из подмножества 6 ---
    r6_present = any(cell % 6 == 0 for row in grids[0] for cell in row)
    # --- Проверяем, есть ли ячейка из подмножества 3 ---
    r3_present = any(cell % 6 == 3 for row in grids[0] for cell in row)
    # --- Проверяем, есть ли 6-е подпространство ---
    sub6_present = any(get_subspace_k(cell) == 6 for row in grids[0] for cell in row)

    with output_area:
        clear_output(wait=True)  # Очищаем предыдущий вывод

        fig = plt.figure(figsize=(30, 10))
        gs = gridspec.GridSpec(3, 6, figure=fig, width_ratios=[1, 0.6, 1, 0.65, 1, 0.6], height_ratios=[0.6, 0.6, 0.6])

        # Графики
        ax_main = fig.add_subplot(gs[0, 0])
        ax_arm = fig.add_subplot(gs[0, 4])
        ax_eo = fig.add_subplot(gs[0, 2])
        ax_sub = fig.add_subplot(gs[1, 0])
        ax_rule = fig.add_subplot(gs[1, 2])
        ax_freeze = fig.add_subplot(gs[1, 4])
        ax_steps = fig.add_subplot(gs[2, 0])

        # Легенды
        legend_main = fig.add_subplot(gs[0, 1])
        legend_arm = fig.add_subplot(gs[0, 5])
        legend_eo = fig.add_subplot(gs[0, 3])
        legend_sub = fig.add_subplot(gs[1, 1])
        legend_rule = fig.add_subplot(gs[1, 3])
        legend_freeze = fig.add_subplot(gs[1, 5])
        legend_steps = fig.add_subplot(gs[2, 1])

        for ax in [ax_arm, ax_eo, ax_sub, ax_rule, ax_main, ax_freeze]:
            ax.clear()
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlim(-0.5, GRID_SIZE_X - 0.5)
            ax.set_ylim(-0.5, GRID_SIZE_Y - 0.5)
            ax.set_aspect('equal')
            ax.axis('off')

        for lax in [legend_arm, legend_eo, legend_sub, legend_rule, legend_main, legend_freeze]:
            lax.clear()
            lax.axis('off')
            lax.set_facecolor('#F9F9F9')


        # --- подмножества (1) ---
        ax_arm.set_title("подмножества (r = n mod 6)", fontsize=10)
        draw_handled(ax_arm, grids[0], get_arm_color, arm_texts, arm_cells)
        if FLAG_FOG:
            draw_fog_of_war(ax_arm, grids[0], player_pos[0], player_pos[1])

        # --- Чётность (2) ---
        ax_eo.set_title("Чётность", fontsize=10)
        draw_handled(ax_eo, grids[0], get_even_odd_color, eo_texts, eo_cells)
        if FLAG_FOG:
            draw_fog_of_war(ax_eo, grids[0], player_pos[0], player_pos[1])

        # --- Подпространства (3) ---
        ax_sub.set_title("Подпространства", fontsize=10)
        draw_handled(ax_sub, grids[0], get_subspace_color, sub_texts, sub_cells)
        if FLAG_FOG:
            draw_fog_of_war(ax_sub, grids[0], player_pos[0], player_pos[1])

        # --- Правила (4) ---
        ax_rule.set_title("Активные правила", fontsize=10)
        draw_handled(ax_rule, grids[0], get_rule_color, combined_texts, combined_cells)
        if FLAG_FOG:
            draw_fog_of_war(ax_rule, grids[0], player_pos[0], player_pos[1])

        # --- Визуализация 5: главный комбинированный график ---
        ax_main.set_title("Главный график: Подпространства + цикл", fontsize=10)
        draw_handled(ax_main, grids[0], get_combined_color, main_texts, main_cells)
        if FLAG_FOG:
            draw_fog_of_war(ax_main, grids[0], player_pos[0], player_pos[1])

        # --- Визуализация 6: заморозка ---
        is_predicted_frozen = predict_frozen_cells(grids[0])
        ax_freeze.set_title("Замороженные ячейки", fontsize=10)
        draw_freeze_map(ax_freeze, grids[0], is_predicted_frozen)
        if FLAG_FOG:
            draw_fog_of_war(ax_freeze, grids[0], player_pos[0], player_pos[1])

        # --- Визуализация 7: шаги до аттрактора ---
        ax_steps.set_title("Шаги до числа 4", fontsize=10)
        draw_steps(ax_steps, grids[0], step_texts, step_cells)
        if FLAG_FOG:
            draw_fog_of_war(ax_steps, grids[0], player_pos[0], player_pos[1])

        # --------------- персонаж -------------------------------------
        current_number = grids[0][player_pos[0]][player_pos[1]]
        if player_circle:
            player_circle.remove()
        if player_text:
            player_text.remove()
        player_circle = plt.Circle((player_pos[1], player_pos[0]), PLAYER_RADIUS, color=PLAYER_COLOR, zorder=5)
        ax_main.add_patch(player_circle)
        fff =''
        if current_number > -1:
            fff = str(current_number)
        player_text = ax_main.text(player_pos[1], player_pos[0], fff,
                                  ha='center', va='center', fontsize=8, color='white', zorder=6)

        # --------------- Кристалл -------------------------------------
        if crystal_coords:
          if grids[0][crystal_coords[0][0]][crystal_coords[0][1]] == STATIC_CELL_CRYSTAL:
            draw_crystal(ax_arm, crystal_coords[0][1], crystal_coords[0][0])
            draw_crystal(ax_eo, crystal_coords[0][1], crystal_coords[0][0])
            draw_crystal(ax_sub, crystal_coords[0][1], crystal_coords[0][0])
            draw_crystal(ax_rule, crystal_coords[0][1], crystal_coords[0][0])
            draw_crystal(ax_main, crystal_coords[0][1], crystal_coords[0][0])
            draw_crystal(ax_freeze, crystal_coords[0][1], crystal_coords[0][0])
            draw_crystal(ax_steps, crystal_coords[0][1], crystal_coords[0][0])

        draw_predator(ax_main, grids[0])
        # ------------------ Легенды -------------------------------------
        def add_legend(ax, elements, title=""):
            ax.clear()
            ax.axis('off')
            ax.set_facecolor('#F9F9F9')
            ax.text(0.05, 0.95, title, fontsize=10, fontweight='bold', transform=ax.transAxes)
            for i, el in enumerate(elements):
                y_pos = 0.9 - i * 0.1
                color = el.get_facecolor()
                ax.add_patch(plt.Rectangle((0.05, y_pos), 0.1, 0.06, facecolor=color, edgecolor='black', transform=ax.transAxes))
                ax.text(0.18, y_pos + 0.01, el.get_label(), fontsize=7, transform=ax.transAxes, va='center')

        # --- Легенда для подмножеств ---
        arm_legend_elements = [Patch(facecolor=ARM_COLORS[r], edgecolor='black', label=f'подмножество {r} (n mod 6)') for r in ARM_COLORS]
        add_legend(legend_arm, arm_legend_elements, "")

        # --- Легенда для заморозки ---
        freeze_legend_elements = [
            Patch(facecolor="#0000AA", edgecolor='black', label='Заморожено'),
            Patch(facecolor="#00AA00", edgecolor='black', label='Доступно'),
        ]
        add_legend(legend_freeze, freeze_legend_elements, "")

        # --- Легенда для количества шагов ---
        steps_legend_elements = [
       #     Patch(facecolor="#FFFFFF", edgecolor='black', label='Остальное'),
       #     Patch(facecolor="#FFD700", edgecolor='black', label='Цикл'),
        ]
        add_legend(legend_steps, steps_legend_elements, "")

        # --- Легенда для чётности ---
        eo_legend_elements = [
            Patch(facecolor=EVEN_ODD_COLORS["odd"], edgecolor='black', label='Нечётное'),
            Patch(facecolor=EVEN_ODD_COLORS["even"], edgecolor='black', label='Чётное'),
            Patch(facecolor=EVEN_ODD_COLORS["cycle"], edgecolor='black', label='Цикл (4→2→1)'),
        ]
        add_legend(legend_eo, eo_legend_elements, "")

        # --- Легенда для основго графика ---
        main_legend_elements = [
            Patch(facecolor="#FFD700", edgecolor='black', label='ЦИКЛ'),
            Patch(facecolor="#F0F0F0", edgecolor='black', label='Четное'),
            Patch(facecolor="#FF6347", edgecolor='black', label='Нечетное, 1 ПП'),
            Patch(facecolor="#32CD32", edgecolor='black', label='Нечетное, 2-6 ПП'),
            Patch(facecolor="#CCCCCC", edgecolor='black', label='Неопределено'),
            Patch(facecolor=COLOR_END_CELL, edgecolor='black', label='Старт/финиш'),
            Patch(facecolor=COLOR_BOOST_REGION, edgecolor='black', label='Ячейка -2 ⇒ усиление'),
            Patch(facecolor=COLOR_EMPTY, edgecolor='black', label='Ячейка -1 ⇒ пустота'),
            Patch(facecolor=COLOR_CRAFT_REGION, edgecolor='black', label='Крафтовая ячейка'),
        ]
        add_legend(legend_main, main_legend_elements, "")

        # --- Легенда для подпространств ---
        subspace_legend_elements = [Patch(facecolor=SUBSPACE_COLORS[k], edgecolor='black', label=f'Подпространство {k}') for k in SUBSPACE_COLORS if k not in ["even", "cycle"]]
        add_legend(legend_sub, subspace_legend_elements, "")

        # --- Легенда для правил ---
        rule_legend_elements = [
            Patch(facecolor="#FF0000", edgecolor='black', label='Обновление при достижении 4'),
            Patch(facecolor="#00FF00", edgecolor='black', label='Спуск ускоряется (рядом магистраль)'),
            Patch(facecolor="#0000FF", edgecolor='black', label='Заморозка (рядом нечетные)'),
            Patch(facecolor="#FFFF00", edgecolor='black', label='Заморозка (рядом обновление)'),
            Patch(facecolor="#FFA500", edgecolor='black', label='2 шага (рядом четные)'),
            Patch(facecolor='#FF6347', edgecolor='black', label='ТОЛЬКО чётные обновляются'),
            Patch(facecolor='#4682B4', edgecolor='black', label='ТОЛЬКО нечётные обновляются'),
            Patch(facecolor='#32CD32', edgecolor='black', label='Чётные числа +2 (Sub6)'),
            Patch(facecolor='#BA55D3', edgecolor='black', label='Заморозка соседей при достижении 4'),
            Patch(facecolor="#FFFFFF", edgecolor='black', label='нет активных правил')
        ]
        add_legend(legend_rule, rule_legend_elements, "")

        plt.show()
#------------------------graph---------------------------------------

# Генерация

In [4]:
global grids, frozen_cells_global  # ОБЯЗАТЕЛЬНО объявляем, что работаем с глобальной переменной
grids = [initialize_grid()]
initialize_graphics_arrays()
frozen_cells_global = set()





#-----------------------------game---------------------------------------------
for key in OPTIONS:
    option_widgets[key].observe(make_checkbox_handler(key), names='value')

# === Создание UI панели ===

# === Кнопки управления игроком ===
control_buttons = widgets.VBox([
    btn_up,
    widgets.HBox([btn_left, btn_stay, btn_right]),
    btn_down,
    btn_shield,
    life_label,
    shield_label,
    craft_label,
    frezze_label,
    back_label,
    crystal_label,
    pres_label,
    info_label,
    step_button,
    btn_frezze,
    btn_back
], layout=widgets.Layout(width='auto', height='auto'))

# === Чекбоксы с опциями ===
option_checkboxes = widgets.VBox(
    [option_widgets[key] for key in OPTIONS],
    layout=widgets.Layout(width='auto')
)

# === Панель управления: левый + правый блоки ===
ui_panel = widgets.HBox([
    control_buttons,
    option_checkboxes
], layout=widgets.Layout(
    display='flex',
    flex_flow='row',
    align_items='flex-start',
    justify_content='space-between',
    width='100%'
))

# === Первичная отрисовка ===
grid_initial = grids[0]

# === Первичная отрисовка с прогнозом заморозок ===
predicted_frozen = predict_frozen_cells(grids[0])
draw_grid(predicted_frozen=predicted_frozen)
#-----------------------------game---------------------------------------------


# Отображение

In [5]:
display(ui_panel, output_area)

Output()